In [1]:
# Ячейка 1: Импорты, установка и база знаний (С ИСПРАВЛЕНИЕМ ДЛЯ MAC)
import os
os.environ["TOKENIZERS_PARALLELISM"] = "false"  # ВАЖНО для Mac!

import subprocess
import sys
import random
import re
from typing import List, Dict, Optional, Tuple

# Устанавливаем всё что нужно
print("=== Установка зависимостей ===")
packages = ["numpy", "pandas", "scikit-learn", "faiss-cpu", "sentence-transformers"]
for pkg in packages:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])
print("=== Зависимости готовы ===\n")

# Импортируем
import numpy as np
import pandas as pd
import faiss
from sentence_transformers import SentenceTransformer
from IPython.display import display, Markdown

# Фиксируем SEED
random.seed(42)
np.random.seed(42)

print(f"NumPy: {np.__version__}")
print(f"Pandas: {pd.__version__}")
print(f"FAISS доступен: {faiss is not None}")

# Создаем папку для артефактов
os.makedirs("./artifacts", exist_ok=True)
print("Папка './artifacts' создана.\n")

# ===== БАЗА ЗНАНИЙ =====
documents: List[Dict[str, str]] = [
    {
        "doc_id": "nlp_01",
        "title": "Что такое NLP",
        "text": "Natural Language Processing (NLP) — это область искусственного интеллекта, которая фокусируется на взаимодействии между компьютерами и человеческим языком. Цель NLP — научить компьютеры понимать, интерпретировать и генерировать человеческий язык осмысленным и полезным способом. Примеры задач NLP включают машинный перевод, анализ тональности, распознавание именованных сущностей и создание чат-ботов."
    },
    {
        "doc_id": "nlp_02",
        "title": "Трансформеры",
        "text": "Трансформеры — это архитектура нейронных сетей, которая произвела революцию в NLP. Представленная в статье 'Attention is All You Need' в 2017 году, она основана на механизме внимания, который позволяет модели взвешивать важность разных частей входного текста при обработке. Трансформеры лежат в основе таких мощных моделей, как BERT, GPT и T5."
    },
    {
        "doc_id": "nlp_03",
        "title": "Механизм внимания",
        "text": "Механизм внимания (Attention Mechanism) — ключевой компонент современных NLP-моделей. Он позволяет модели фокусироваться на наиболее релевантных частях входных данных при формировании выхода. В отличие от рекуррентных сетей (RNN), которые обрабатывают текст последовательно, внимание может параллельно оценивать связи между всеми словами в предложении, что значительно ускоряет обучение и улучшает понимание контекста."
    },
    {
        "doc_id": "nlp_04",
        "title": "Большие языковые модели (LLM)",
        "text": "Большие языковые модели (LLM), такие как GPT-4, Claude и Llama, — это нейронные сети с миллиардами параметров, обученные на огромных массивах текстовых данных. Они демонстрируют удивительные способности к обобщению и могут решать широкий круг задач без специального дообучения (zero-shot learning). LLM способны генерировать связный текст, переводить языки, писать код, отвечать на вопросы и многое другое."
    },
    {
        "doc_id": "nlp_05",
        "title": "Токенизация",
        "text": "Токенизация — это процесс разбиения текста на более мелкие единицы, называемые токенами. Токенами могут быть слова, части слов (подслова) или отдельные символы. Это первый и обязательный шаг перед подачей текста в NLP-модель. Современные модели, такие как GPT, часто используют подсловную токенизацию (например, Byte-Pair Encoding, BPE), чтобы эффективно обрабатывать редкие и неизвестные слова."
    },
    {
        "doc_id": "nlp_06",
        "title": "Промпт-инжиниринг",
        "text": "Промпт-инжиниринг — это искусство и наука создания эффективных запросов (промптов) для больших языковых моделей. Хорошо сформулированный промпт может значительно улучшить качество ответа модели, направив её в нужное русло. Методы включают few-shot prompting (предоставление примеров), chain-of-thought (просьба рассуждать по шагам) и указание роли модели."
    }
]

print(f"Количество документов: {len(documents)}")
for doc in documents:
    print(f"  - {doc['doc_id']}: {doc['title']}")

=== Установка зависимостей ===
=== Зависимости готовы ===



/Library/Frameworks/Python.framework/Versions/3.11/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


NumPy: 2.4.4
Pandas: 3.0.2
FAISS доступен: True
Папка './artifacts' создана.

Количество документов: 6
  - nlp_01: Что такое NLP
  - nlp_02: Трансформеры
  - nlp_03: Механизм внимания
  - nlp_04: Большие языковые модели (LLM)
  - nlp_05: Токенизация
  - nlp_06: Промпт-инжиниринг


In [2]:
# Ячейка 2: Чанкинг документов
def chunk_text(text: str, chunk_size: int = 250, overlap: int = 40) -> List[str]:
    chunks = []
    start = 0
    text_len = len(text)
    while start < text_len:
        end = start + chunk_size
        chunk = text[start:end]
        chunks.append(chunk)
        if end >= text_len:
            break
        start = end - overlap
    return chunks

CHUNK_SIZE = 250
OVERLAP = 40

chunk_rows = []
for doc in documents:
    chunks = chunk_text(doc["text"], chunk_size=CHUNK_SIZE, overlap=OVERLAP)
    for i, chunk_text_val in enumerate(chunks):
        chunk_rows.append({
            "doc_id": doc["doc_id"],
            "title": doc["title"],
            "chunk_id": f"{doc['doc_id']}_chunk_{i:03d}",
            "chunk_text": chunk_text_val
        })

chunks_df = pd.DataFrame(chunk_rows)
print(f"Всего создано чанков: {len(chunks_df)}")
print("Первые 3 чанка:")
for i in range(min(3, len(chunks_df))):
    print(f"  {i+1}. {chunks_df.iloc[i]['chunk_id']}: {chunks_df.iloc[i]['chunk_text'][:80]}...")

Всего создано чанков: 12
Первые 3 чанка:
  1. nlp_01_chunk_000: Natural Language Processing (NLP) — это область искусственного интеллекта, котор...
  2. nlp_01_chunk_001: ть и генерировать человеческий язык осмысленным и полезным способом. Примеры зад...
  3. nlp_02_chunk_000: Трансформеры — это архитектура нейронных сетей, которая произвела революцию в NL...


In [3]:
# Ячейка 3: Эмбеддинги и индекс FAISS (исправлено для Mac)
import warnings
warnings.filterwarnings('ignore')

print("Загружаем модель...")
model = SentenceTransformer('paraphrase-multilingual-MiniLM-L12-v2')
print("Модель загружена.")

chunk_texts = chunks_df["chunk_text"].tolist()
print(f"Чанков: {len(chunk_texts)}")

# Кодируем по одному
chunk_embeddings = []
for i, text in enumerate(chunk_texts):
    emb = model.encode([text], normalize_embeddings=True)
    chunk_embeddings.append(emb[0])

chunk_embeddings = np.array(chunk_embeddings, dtype=np.float32)
print(f"Форма: {chunk_embeddings.shape}")

index = faiss.IndexFlatIP(chunk_embeddings.shape[1])
index.add(chunk_embeddings)
print("Индекс FAISS создан.")

Загружаем модель...


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 18617.23it/s]
BertModel LOAD REPORT from: sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Модель загружена.
Чанков: 12
Форма: (12, 384)
Индекс FAISS создан.


In [4]:
# Ячейка 4: Функция поиска
def search(query: str, top_k: int = 3) -> pd.DataFrame:
    query_embedding = model.encode([query], normalize_embeddings=True)
    scores, indices = index.search(query_embedding, top_k)

    results = []
    for rank, (score, idx) in enumerate(zip(scores[0], indices[0]), start=1):
        chunk = chunks_df.iloc[idx].to_dict()
        chunk["rank"] = rank
        chunk["score"] = score
        results.append(chunk)

    return pd.DataFrame(results)[["rank", "score", "doc_id", "title", "chunk_id", "chunk_text"]]

# Проверка
sample_query = "Как работает механизм внимания?"
print(f"Запрос: {sample_query}")
result = search(sample_query, top_k=3)
for i, row in result.iterrows():
    print(f"  {row['rank']}. {row['doc_id']} (score: {row['score']:.3f})")

Запрос: Как работает механизм внимания?
  1. nlp_03 (score: 0.739)
  2. nlp_02 (score: 0.566)
  3. nlp_03 (score: 0.448)


In [5]:
# Ячейка 5: Контрольные запросы и оценка retrieval
# Бенчмарк с ожидаемыми документами
benchmark_queries = [
    {"query_id": "q01", "query": "Что такое NLP?", "relevant_doc_id": "nlp_01"},
    {"query_id": "q02", "query": "Какая архитектура нейросетей произвела революцию в NLP?", "relevant_doc_id": "nlp_02"},
    {"query_id": "q03", "query": "Что позволяет модели взвешивать важность слов?", "relevant_doc_id": "nlp_03"},
    {"query_id": "q04", "query": "Приведи примеры больших языковых моделей.", "relevant_doc_id": "nlp_04"},
    {"query_id": "q05", "query": "Что такое токенизация?", "relevant_doc_id": "nlp_05"},
    {"query_id": "q06", "query": "Как создавать эффективные запросы к LLM?", "relevant_doc_id": "nlp_06"},
]

def evaluate_retrieval(queries, k=3):
    eval_results = []
    for item in queries:
        query = item["query"]
        expected_source = item["relevant_doc_id"]
        results_df = search(query, top_k=k)
        
        retrieved_sources = ", ".join(results_df["doc_id"].values)
        hit_at_k = 1 if expected_source in results_df["doc_id"].values else 0
        recall_at_k = hit_at_k  # Для одного релевантного документа
        
        rank = results_df[results_df["doc_id"] == expected_source]["rank"].values
        rank_of_first_relevant = rank[0] if len(rank) > 0 else None
        
        eval_results.append({
            "query": query,
            "expected_source": expected_source,
            "retrieved_sources": retrieved_sources,
            "hit_at_k": hit_at_k,
            f"hit@{k}_internal": hit_at_k,
            f"recall@{k}_internal": recall_at_k,
            "rank_of_first_relevant": rank_of_first_relevant
        })
    return pd.DataFrame(eval_results)

# Запуск оценки
TOP_K = 3
eval_df = evaluate_retrieval(benchmark_queries, k=TOP_K)
print("\n=== Результаты оценки retrieval ===")
display(eval_df[["query", "expected_source", "retrieved_sources", "hit_at_k", "rank_of_first_relevant"]])

# Расчёт метрик
mean_hit = eval_df[f"hit@{TOP_K}_internal"].mean()
mean_recall = eval_df[f"recall@{TOP_K}_internal"].mean()
print(f"\nСредний hit@{TOP_K}: {mean_hit:.2f}")
print(f"Средний recall@{TOP_K}: {mean_recall:.2f}")

# Сохраняем с ОБЯЗАТЕЛЬНЫМИ колонками
eval_df_to_save = eval_df[["query", "expected_source", "retrieved_sources", "hit_at_k"]]
eval_df_to_save.to_csv("./artifacts/retrieval_eval.csv", index=False)
print("\nФайл './artifacts/retrieval_eval.csv' сохранен.")


=== Результаты оценки retrieval ===


,query,expected_source,retrieved_sources,hit_at_k,rank_of_first_relevant
0,Что такое NLP?,nlp_01,"nlp_01, nlp_03, nlp_02",1,1
1,Какая архитектура нейросетей произвела революц...,nlp_02,"nlp_02, nlp_03, nlp_01",1,1
2,Что позволяет модели взвешивать важность слов?,nlp_03,"nlp_01, nlp_03, nlp_06",1,2
3,Приведи примеры больших языковых моделей.,nlp_04,"nlp_04, nlp_01, nlp_06",1,1
4,Что такое токенизация?,nlp_05,"nlp_05, nlp_05, nlp_06",1,1
5,Как создавать эффективные запросы к LLM?,nlp_06,"nlp_04, nlp_04, nlp_06",1,3



Средний hit@3: 1.00
Средний recall@3: 1.00

Файл './artifacts/retrieval_eval.csv' сохранен.


In [6]:
# Ячейка 6: Эксперимент с параметром chunk_size
def run_experiment(chunk_size, k=3):
    print(f"\n--- Эксперимент с chunk_size={chunk_size} ---")
    
    # Временные чанки
    temp_chunks = []
    for doc in documents:
        chunks = chunk_text(doc["text"], chunk_size=chunk_size, overlap=40)
        for i, chunk_text_val in enumerate(chunks):
            temp_chunks.append({
                "doc_id": doc["doc_id"],
                "title": doc["title"],
                "chunk_id": f"{doc['doc_id']}_chunk_{i:03d}",
                "chunk_text": chunk_text_val
            })
    temp_chunks_df = pd.DataFrame(temp_chunks)
    temp_texts = temp_chunks_df["chunk_text"].tolist()
    
    # Эмбеддинги
    temp_embeddings = []
    for text in temp_texts:
        emb = model.encode([text], normalize_embeddings=True, show_progress_bar=False)
        temp_embeddings.append(emb[0])
    temp_embeddings = np.array(temp_embeddings, dtype=np.float32)
    
    # Индекс
    temp_index = faiss.IndexFlatIP(temp_embeddings.shape[1])
    temp_index.add(temp_embeddings)
    
    # Подмена глобальных переменных
    global chunks_df, index
    original_chunks_df, original_index = chunks_df, index
    chunks_df, index = temp_chunks_df, temp_index
    
    # Оценка
    eval_df = evaluate_retrieval(benchmark_queries, k=k)
    
    # Возврат
    chunks_df, index = original_chunks_df, original_index
    
    return {
        "chunk_size": chunk_size,
        "num_chunks": len(temp_chunks_df),
        f"mean_hit@{k}": eval_df[f"hit@{k}_internal"].mean(),
        f"mean_recall@{k}": eval_df[f"recall@{k}_internal"].mean()
    }

# Запуск экспериментов
exp_results = [run_experiment(200, k=TOP_K), run_experiment(350, k=TOP_K)]
exp_df = pd.DataFrame(exp_results)
print("\n=== Результаты эксперимента ===")
display(exp_df)


--- Эксперимент с chunk_size=200 ---

--- Эксперимент с chunk_size=350 ---

=== Результаты эксперимента ===


,chunk_size,num_chunks,mean_hit@3,mean_recall@3
0,200,16,0.833333,0.833333
1,350,11,1.000000,1.000000


In [7]:
# Ячейка 7: Обновление базы знаний и переиндексация
# Запросы для сравнения
queries_for_comparison = [
    "Что такое RAG в контексте LLM?",
    "Как работает RLHF?",
    "Зачем нужен fine-tuning?"
]

print("=== РЕЗУЛЬТАТЫ ДО ОБНОВЛЕНИЯ ===")
before_results = {}
for q in queries_for_comparison:
    res_df = search(q, top_k=3)
    before_results[q] = ", ".join(res_df["doc_id"].values)
    print(f"\nЗапрос: {q}")
    print(f"Найденные ID: {before_results[q]}")

# Добавляем новые документы
new_documents = [
    {
        "doc_id": "nlp_07",
        "title": "RAG (Retrieval-Augmented Generation)",
        "text": "RAG (Retrieval-Augmented Generation) — это подход, который объединяет поиск информации и генеративные модели. Сначала система извлекает релевантные документы из внешней базы знаний, а затем языковая модель использует эту информацию для генерации более точного и фактически обоснованного ответа. RAG особенно полезен для задач, требующих актуальных знаний, которые модель могла не видеть во время обучения."
    },
    {
        "doc_id": "nlp_08",
        "title": "RLHF (Reinforcement Learning from Human Feedback)",
        "text": "RLHF (Reinforcement Learning from Human Feedback) — это метод обучения языковых моделей, при котором люди оценивают ответы модели, и эта обратная связь используется для улучшения модели с помощью обучения с подкреплением. RLHF является ключевым компонентом при создании полезных и безопасных ассистентов, таких как ChatGPT."
    },
    {
        "doc_id": "nlp_09",
        "title": "Fine-tuning языковых моделей",
        "text": "Fine-tuning (тонкая настройка) — это процесс дополнительного обучения предобученной языковой модели на небольшом специализированном наборе данных. Это позволяет адаптировать модель для конкретной задачи или предметной области, значительно улучшая её производительность без необходимости обучать модель с нуля."
    }
]

documents.extend(new_documents)
print("\n\nДокументы добавлены. Переиндексация...")

# Переиндексация
chunk_rows = []
for doc in documents:
    chunks = chunk_text(doc["text"], chunk_size=CHUNK_SIZE, overlap=OVERLAP)
    for i, chunk_text_val in enumerate(chunks):
        chunk_rows.append({
            "doc_id": doc["doc_id"],
            "title": doc["title"],
            "chunk_id": f"{doc['doc_id']}_chunk_{i:03d}",
            "chunk_text": chunk_text_val
        })
chunks_df = pd.DataFrame(chunk_rows)
chunk_texts = chunks_df["chunk_text"].tolist()

chunk_embeddings = []
for text in chunk_texts:
    emb = model.encode([text], normalize_embeddings=True, show_progress_bar=False)
    chunk_embeddings.append(emb[0])
chunk_embeddings = np.array(chunk_embeddings, dtype=np.float32)

index = faiss.IndexFlatIP(chunk_embeddings.shape[1])
index.add(chunk_embeddings)
print("Переиндексация завершена.")

print("\n=== РЕЗУЛЬТАТЫ ПОСЛЕ ОБНОВЛЕНИЯ ===")
comparison_data = []
for q in queries_for_comparison:
    res_df = search(q, top_k=3)
    after_doc_ids = ", ".join(res_df["doc_id"].values)
    print(f"\nЗапрос: {q}")
    print(f"Найденные ID: {after_doc_ids}")
    
    comparison_data.append({
        "query": q,
        "before_retrieved_sources": before_results[q],
        "after_retrieved_sources": after_doc_ids,
        "changed": before_results[q] != after_doc_ids
    })

comparison_df = pd.DataFrame(comparison_data)
comparison_df.to_csv("./artifacts/retrieval_before_after_update.csv", index=False)
print("\nФайл './artifacts/retrieval_before_after_update.csv' сохранен.")
display(comparison_df)

=== РЕЗУЛЬТАТЫ ДО ОБНОВЛЕНИЯ ===

Запрос: Что такое RAG в контексте LLM?
Найденные ID: nlp_04, nlp_04, nlp_03

Запрос: Как работает RLHF?
Найденные ID: nlp_04, nlp_04, nlp_03

Запрос: Зачем нужен fine-tuning?
Найденные ID: nlp_06, nlp_06, nlp_03


Документы добавлены. Переиндексация...
Переиндексация завершена.

=== РЕЗУЛЬТАТЫ ПОСЛЕ ОБНОВЛЕНИЯ ===

Запрос: Что такое RAG в контексте LLM?
Найденные ID: nlp_07, nlp_07, nlp_04

Запрос: Как работает RLHF?
Найденные ID: nlp_08, nlp_08, nlp_04

Запрос: Зачем нужен fine-tuning?
Найденные ID: nlp_09, nlp_07, nlp_06

Файл './artifacts/retrieval_before_after_update.csv' сохранен.


,query,before_retrieved_sources,after_retrieved_sources,changed
0,Что такое RAG в контексте LLM?,"nlp_04, nlp_04, nlp_03","nlp_07, nlp_07, nlp_04",True
1,Как работает RLHF?,"nlp_04, nlp_04, nlp_03","nlp_08, nlp_08, nlp_04",True
2,Зачем нужен fine-tuning?,"nlp_06, nlp_06, nlp_03","nlp_09, nlp_07, nlp_06",True


In [8]:
# Ячейка 8: Mini-RAG
def split_into_sentences(text):
    return re.split(r'(?<=[.!?])\s+', text)

def mini_rag_answer(query: str, top_k: int = 3) -> Dict:
    retrieved_df = search(query, top_k=top_k)
    context = "\n\n".join([f"[{row['doc_id']}] {row['chunk_text']}" for _, row in retrieved_df.iterrows()])
    
    # Ответ — последнее предложение из топ-1 чанка
    top_chunk_text = retrieved_df.iloc[0]['chunk_text']
    sentences = split_into_sentences(top_chunk_text)
    answer = sentences[-1] if sentences else top_chunk_text
    
    return {
        "question": query,
        "answer": answer,
        "retrieved_sources": ", ".join(retrieved_df["doc_id"].unique())
    }

# Примеры
rag_examples = []
test_questions = [
    "Что такое NLP?",
    "Как работают трансформеры?",
    "Как улучшить промпт для LLM?"
]

print("=== Примеры работы mini-RAG ===\n")
for q in test_questions:
    rag_result = mini_rag_answer(q)
    print(f"Вопрос: {q}")
    print(f"Ответ: {rag_result['answer']}")
    print(f"Источники: {rag_result['retrieved_sources']}")
    print("-" * 50)
    
    rag_examples.append({
        "question": q,
        "answer": rag_result['answer'],
        "retrieved_sources": rag_result['retrieved_sources']
    })

rag_examples_df = pd.DataFrame(rag_examples)
rag_examples_df.to_csv("./artifacts/rag_examples.csv", index=False)
print("\nФайл './artifacts/rag_examples.csv' сохранен.")

=== Примеры работы mini-RAG ===

Вопрос: Что такое NLP?
Ответ: Цель NLP — научить компьютеры понимать, интерпретировать и генерировать человеческий язык осмы
Источники: nlp_01, nlp_03, nlp_02
--------------------------------------------------
Вопрос: Как работают трансформеры?
Ответ: Представленная в статье 'Attention is All You Need' в 2017 году, она основана на механизме внимания, который позволяет модели взвешивать важность разных частей входног
Источники: nlp_02, nlp_06
--------------------------------------------------
Вопрос: Как улучшить промпт для LLM?
Ответ: LLM способны генерировать связный текст, переводить языки, писать код, отвечать на вопросы и многое другое.
Источники: nlp_04, nlp_06, nlp_09
--------------------------------------------------

Файл './artifacts/rag_examples.csv' сохранен.
